# [LAB] Harness 보강 실습: 사람에게 묻고 승인받는 Deep Agent

`create_deep_agent`로 만든 Agent에 안전장치를 하나씩 붙여 봅니다. 모델에게 부탁하는 대신 프레임워크가 지키게 하는 장치입니다.

- **2장** 메일을 보내기 전에 멈추고, 사람이 승인·수정·거절합니다.
- **3장** Agent가 선택지를 주고 사람에게 묻습니다.
- **4장** 2·3장을 터미널에서 화살표 메뉴로 해 봅니다.
- **5장** Agent가 고칠 수 없는 폴더를 정합니다.
- **6장** 조사를 하위 Agent에게 맡깁니다.

장마다 첫 셀(접힌 준비 코드)부터 차례로 실행합니다. Agent를 실행하는 셀은 수십 초 걸리고, 답은 실행할 때마다 조금씩 다릅니다. 메일 도구는 연습용이라 실제로 보내지 않습니다.

**목차**

- 1. 실습 방법
- 2. 보내기 전에 승인받기
  - 2.1 Agent 만들기
  - 2.2 요청하고 멈춘 곳 보기
  - 2.3 결정 보내기
- 3. Agent가 사람에게 묻기
  - 3.1 Agent 만들기
  - 3.2 요청하고 질문 받기
  - 3.3 답하고 승인하기
- 4. 터미널에서 메뉴로 답하기
  - 4.1 Agent 파일 쓰기
- 5. 고칠 수 있는 파일 정하기
  - 5.1 Agent 만들기
  - 5.2 파일 고쳐 달라고 하기
- 6. 조사는 하위 Agent에게 맡기기
  - 6.1 하위 Agent 만들고 맡기기
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 보내기 전에 승인받기

`interrupt_on`에 도구 이름을 적으면, Agent가 그 도구를 쓰기 직전에 멈춥니다. 사람은 셋 중 하나로 답합니다.

- `approve`: 그대로 보낸다
- `edit`: 고쳐서 보낸다
- `reject`: 보내지 않는다(이유를 Agent에게 전한다)

멈춘 자리는 체크포인터(`MemorySaver`)가 기억합니다. `Command(resume=...)`로 결정을 보내면 그 자리부터 이어서 실행합니다.

**왜 배우나:** "보내기 전에 물어봐"를 프롬프트에 적으면 모델이 잊을 수 있습니다. `interrupt_on`은 반드시 멈춥니다.

준비 코드입니다. 연습용 메일 도구 `send_email`을 만듭니다. 접혀 있으니 실행만 합니다.

In [ ]:
import uuid

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from config.llm import get_llm

load_dotenv()


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """메일을 보낸다."""
    return f"메일을 보냈습니다. 제목: {subject} | 본문: {body}"  # 연습용: 실제로 보내지 않는다

### 2.1 Agent 만들기

In [ ]:
# ── Agent 만들기 ──
agent = create_deep_agent(
    model=get_llm(),
    tools=[send_email],
    system_prompt="너는 홍길동 프로의 메일 비서다. 메일은 짧게 쓰고 끝에 '홍길동 드림'을 붙인다.",
    # send_email을 쓰기 직전에 멈추고, 사람의 결정(승인·수정·거절)을 기다린다
    interrupt_on={"send_email": {"allowed_decisions": ["approve", "edit", "reject"]}},
    checkpointer=MemorySaver(),  # 멈춘 자리를 기억한다
)

### 2.2 요청하고 멈춘 곳 보기

In [ ]:
# ── 요청하고 멈춘 곳 보기 ──
config = {"configurable": {"thread_id": str(uuid.uuid4())}}  # 실행할 때마다 새 대화
request = "김민수 책임(minsu.kim@partner.example.com)에게 내일 회의에 10분 늦는다고 메일 보내 줘."
result = agent.invoke({"messages": [("user", request)]}, config, version="v2")

# Agent가 멈췄습니다. 보내려던 메일을 봅니다
email = result.interrupts[0].value["action_requests"][0]["args"]
print(email)

### 2.3 결정 보내기

**✍️ CHANGE_HERE: 결정을 reject나 edit로 바꾸면 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `{"type": "approve"}` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

보내지 않기
```text
{"type": "reject", "message": "보내지 마세요. 보내지 않았다고만 답하세요."}
```

본문을 고쳐 보내기
```text
{"type": "edit", "edited_action": {"name": "send_email", "args": {**email, "body": "10분 늦습니다. 죄송합니다. 홍길동 드림"}}}
```

값을 바꿀 때마다 위의 **요청하고 멈춘 곳 보기** 셀부터 다시 실행해 새로 멈춘 뒤, 이 셀을 실행합니다. 다 봤으면 `{"type": "approve"}`로 되돌립니다.

In [ ]:
# ── 결정 보내기 ──
# ✍️ CHANGE_HERE: 결정을 reject나 edit로 바꾸면 어떻게 될까요?
decision = {"type": "approve"}

result = agent.invoke(Command(resume={"decisions": [decision]}), config, version="v2")
for m in result.value["messages"]:
    print(f"[{m.type}] {m.content[:100]}")

<details><summary>무슨 일이 생기나</summary>

`reject`: `[tool]` 줄이 `` User rejected the tool call for `send_email` with reason: 보내지 마세요. … ``로 바뀌고, 마지막 `[ai]`가 `보내지 않았습니다.`라고 답했습니다. 메일 도구는 실행되지 않았습니다.

`edit`: `[tool]` 줄이 `Note: a human reviewer replaced this tool call …`로 시작합니다. 모델이 쓴 본문 대신 사람이 고친 본문(`10분 늦습니다. 죄송합니다. 홍길동 드림`)으로 실행되었다는 뜻입니다(2026-10-21 실행).

</details>

**볼 지점**
- 요청 셀은 메일을 보내지 않고 멈춥니다. 출력은 Agent가 **보내려던** 메일(받는 사람·제목·본문)입니다.
- `approve`를 보내야 `[tool]` 줄에 `메일을 보냈습니다`가 나옵니다. 그 전까지 도구는 실행되지 않았습니다.

## 3. Agent가 사람에게 묻기

질문 도구 `ask_user`를 더합니다. 질문(`question`)과 선택지(`choices`)를 받는 도구입니다.

- `allowed_decisions`로 도구마다 받을 결정을 정합니다. `ask_user`는 `respond`(사람이 답하기)만, `send_email`은 승인·거절만 받습니다.
- `ask_user`는 실제로 실행되지 않습니다. 사람이 고른 답이 도구 결과 자리에 그대로 들어갑니다.

**왜 배우나:** 질문을 도구로 만들면 Agent가 물을 때마다 반드시 멈추고, 답은 늘 선택지 중 하나입니다. 그래서 4장처럼 메뉴나 버튼으로 보여 줄 수 있습니다.

준비 코드입니다. 질문 도구 `ask_user`와 메일 도구 `send_email`을 만듭니다. 펼치지 않고 실행만 합니다.

In [ ]:
import uuid

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from config.llm import get_llm

load_dotenv()


@tool
def ask_user(question: str, choices: list[str]) -> str:
    """사용자에게 선택지를 보여 주고 하나를 고르게 한다."""
    return ""  # 실행되지 않는다. 사람이 고른 답이 이 자리에 들어간다


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """메일을 보낸다."""
    return f"메일을 보냈습니다. 제목: {subject} | 본문: {body}"  # 연습용: 실제로 보내지 않는다

### 3.1 Agent 만들기

In [ ]:
# ── Agent 만들기 ──
agent = create_deep_agent(
    model=get_llm(),
    tools=[ask_user, send_email],
    system_prompt=(
        "너는 홍길동 프로의 메일 비서다. 사용자가 정해야 할 것이 있으면 메일을 쓰기 전에 ask_user로 선택지를 주고 묻는다. "
        "메일은 홍길동 프로가 보내는 글로 짧게 쓰고 끝에 '홍길동 드림'을 붙인다."
    ),
    interrupt_on={
        # 묻는 도구는 사람이 답하기(respond)만, 보내는 도구는 승인·거절만 받는다
        "ask_user": {"allowed_decisions": ["respond"]},
        "send_email": {"allowed_decisions": ["approve", "reject"]},
    },
    checkpointer=MemorySaver(),
)

### 3.2 요청하고 질문 받기

In [ ]:
# ── 요청하고 질문 받기 ──
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
request = (
    "김민수 책임(minsu.kim@partner.example.com)에게 내가 내일 오후 회의에 못 간다고 메일 보내 줘. "
    "다음 주로 미룰지 온라인으로 바꿀지는 먼저 나한테 물어봐."
)
result = agent.invoke({"messages": [("user", request)]}, config, version="v2")

# Agent가 멈췄습니다. 질문과 선택지를 봅니다
question = result.interrupts[0].value["action_requests"][0]["args"]
print(question)

### 3.3 답하고 승인하기

**✍️ CHANGE_HERE: 두 번째 선택지를 고르면 메일은 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `[0]` 자리에 복사해 붙여 넣기

```text
[1]
```

위의 **요청하고 질문 받기** 셀부터 다시 실행한 뒤 이 셀을 실행합니다. `[0]`은 첫 번째, `[1]`은 두 번째 선택지입니다. 다 봤으면 `[0]`으로 되돌립니다.

In [ ]:
# ── 답하고 승인하기 ──
# ✍️ CHANGE_HERE: 두 번째 선택지를 고르면 메일은 어떻게 달라질까요?
answer = question["choices"][0]

# 고른 답을 보내면 Agent가 메일을 쓰고, 보내기 전에 다시 멈춘다
decision = {"type": "respond", "message": answer}
result = agent.invoke(Command(resume={"decisions": [decision]}), config, version="v2")
print("메일 초안:", result.interrupts[0].value["action_requests"][0]["args"]["body"])

# 초안을 승인한다
decision = {"type": "approve"}
result = agent.invoke(Command(resume={"decisions": [decision]}), config, version="v2")
for m in result.value["messages"]:
    print(f"[{m.type}] {m.content[:100]}")

<details><summary>무슨 일이 생기나</summary>

이번 실행의 선택지는 `['다음 주로 미룬다', '온라인 회의로 변경한다']`였습니다. `[0]`이면 메일 제목이 `내일 오후 회의 불참 및 일정 조정 안내`, `[1]`이면 `내일 오후 회의 불참 및 온라인 회의 제안`이 되고 본문도 온라인 회의를 제안하는 내용으로 바뀌었습니다(2026-10-21 실행). 두 경우 모두 `ask_user`의 `[tool]` 줄에는 고른 선택지 글이 그대로 들어 있었습니다.

</details>

**볼 지점**
- 요청 셀의 출력은 Agent의 질문과 선택지입니다. 메일은 아직 쓰지 않았습니다.
- 마지막 대화에서 첫 `[tool]` 줄이 `ask_user`의 결과입니다. 함수 몸통의 빈 글이 아니라 고른 답이 들어 있습니다.
- 이 실습은 질문에서 한 번, 메일에서 한 번, 모두 두 번 멈춥니다.

## 4. 터미널에서 메뉴로 답하기

2·3장을 합친 Agent를 파일로 만들어 터미널에서 실행합니다. 화살표 메뉴는 노트북 셀 안에서는 동작하지 않기 때문입니다.

- `questionary`는 터미널에 메뉴를 그려 주는 라이브러리입니다. `select`는 화살표 메뉴, `confirm`은 예/아니요, `text`는 글 입력입니다.
- `while result.interrupts:` 반복문이 Agent가 멈출 때마다 메뉴로 답을 받아 이어서 실행합니다. 몇 번 멈추든 같은 코드가 받습니다.
- 메일을 거절하면서 고칠 점을 적으면, Agent가 다시 써서 또 물어봅니다.

**왜 배우나:** 실제 Agent 앱은 이 반복문 위에 화면(버튼·메신저)을 얹습니다. 메뉴만 바꾸면 Agent 코드는 그대로입니다.

### 4.1 Agent 파일 쓰기

In [ ]:
%%writefile hitl_agent.py
import uuid

import questionary
from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from config.llm import get_llm

load_dotenv()


@tool
def ask_user(question: str, choices: list[str]) -> str:
    """사용자에게 선택지를 보여 주고 하나를 고르게 한다."""
    return ""


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """메일을 보낸다."""
    return f"메일을 보냈습니다. 제목: {subject}"  # 연습용: 실제로 보내지 않는다


agent = create_deep_agent(
    model=get_llm(),
    tools=[ask_user, send_email],
    system_prompt=(
        "너는 홍길동 프로의 메일 비서다. 사용자가 정해야 할 것이 있으면 메일을 쓰기 전에 ask_user로 선택지를 주고 묻는다. "
        "메일은 홍길동 프로가 보내는 글로 짧게 쓰고 끝에 '홍길동 드림'을 붙인다. 메일이 거절되면 이유에 맞춰 다시 써서 보낸다."
    ),
    interrupt_on={
        "ask_user": {"allowed_decisions": ["respond"]},
        "send_email": {"allowed_decisions": ["approve", "reject"]},
    },
    checkpointer=MemorySaver(),
)
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

request = questionary.text(
    "Agent에게 보낼 요청:",
    default="김민수 책임(minsu.kim@partner.example.com)에게 내가 내일 오후 회의에 못 간다고 메일 보내 줘. "
    "다음 주로 미룰지 온라인으로 바꿀지는 먼저 나한테 물어봐.",
).ask()
print("Agent가 일하는 중입니다...")
result = agent.invoke({"messages": [("user", request)]}, config, version="v2")

# Agent가 멈출 때마다 메뉴로 답하고 이어서 실행한다. 몇 번 멈추든 이 반복문이 받는다
while result.interrupts:
    action = result.interrupts[0].value["action_requests"][0]
    if action["name"] == "ask_user":  # 질문: 화살표 메뉴로 고르기
        answer = questionary.select(
            action["args"]["question"], choices=action["args"]["choices"]
        ).ask()
        decision = {"type": "respond", "message": answer}
    else:  # 메일: 보여 주고 승인 또는 거절
        print("\n" + action["args"]["body"] + "\n")
        if questionary.confirm("이 메일을 보낼까요?").ask():
            decision = {"type": "approve"}
        else:
            reason = questionary.text("고칠 점(Agent에게 전해집니다):").ask()
            decision = {"type": "reject", "message": reason}
    print("Agent가 일하는 중입니다...")
    result = agent.invoke(
        Command(resume={"decisions": [decision]}), config, version="v2"
    )

print("\n" + result.value["messages"][-1].content)

위 셀을 실행하면 이 폴더에 `hitl_agent.py`가 만들어집니다. 터미널에서 실행합니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run --with questionary python course/Chapter02/notebooks/hitl_agent.py
```

3. 요청이 채워져 있으니 Enter → 질문은 ↑↓로 고르고 Enter → 메일은 한 번 `n`을 눌러 고칠 점을 적고, 다시 써 온 메일에는 `y`를 누릅니다.
4. 그만두려면 **Ctrl+C**를 누릅니다.

**볼 지점**
- 고칠 점을 적고 거절하면 Agent가 그 말대로 다시 쓴 메일을 보여 줍니다. 이번 실행에서는 "더 짧게 두 줄로 써 주세요"라고 적자 두 줄짜리 메일로 다시 써 왔습니다.
- 질문 문장과 선택지는 모델이 정해서 실행마다 다릅니다.

## 5. 고칠 수 있는 파일 정하기

Deep Agent는 작업 폴더의 파일을 읽고 쓰는 도구를 기본으로 가집니다. `FilesystemPermission`으로 어느 경로에 무엇을 할 수 있는지 정합니다.

- `paths=["/reference/**"]`: 작업 폴더 `harness_plus` 안의 `reference` 폴더 전부
- `mode`: `allow`(허용) · `deny`(막음) · `interrupt`(사람 승인을 기다림)

처음 값은 `deny`입니다. Agent에게 이 폴더의 규칙 파일에 한 줄을 더해 달라고 합니다.

**왜 배우나:** "이 파일은 고치지 마"를 프롬프트에 적는 것보다 도구가 쓰지 못하게 막는 편이 확실합니다.

준비 코드입니다. Agent의 작업 폴더 `harness_plus`를 만듭니다. 실행만 하면 됩니다.

In [ ]:
import uuid
from pathlib import Path

from deepagents import FilesystemPermission, create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

from config.llm import get_llm

load_dotenv()

RULES = Path(
    "harness_plus/reference/rules.md"
)  # Agent에게는 /reference/rules.md로 보인다
RULES.parent.mkdir(parents=True, exist_ok=True)

### 5.1 Agent 만들기

**✍️ CHANGE_HERE: mode를 interrupt나 allow로 바꾸면 파일은 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `deny` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

쓰기 전에 사람 승인 기다리기
```text
interrupt
```

막지 않기
```text
allow
```

값마다 이 셀과 아래 셀을 차례로 다시 실행합니다. 아래 셀은 매번 파일을 처음 내용으로 되돌리고 시작합니다. 다 봤으면 `deny`로 되돌립니다.

In [ ]:
# ── Agent 만들기 ──
# 규칙: /reference/ 아래 파일에 쓰기(write)를 mode에 따라 허용·막음·승인 대기
# ✍️ CHANGE_HERE: mode를 interrupt나 allow로 바꾸면 파일은 어떻게 될까요?
rule = FilesystemPermission(operations=["write"], paths=["/reference/**"], mode="deny")

agent = create_deep_agent(
    model=get_llm(),
    backend=FilesystemBackend(root_dir="harness_plus", virtual_mode=True),  # 작업 폴더
    permissions=[rule],
    checkpointer=MemorySaver(),
)

<details><summary>무슨 일이 생기나</summary>

`interrupt`: `멈췄나요?: True`가 나오고 Agent의 답은 비어 있으며 파일은 그대로였습니다. 쓰기 직전에 사람 승인을 기다리며 멈춘 것입니다.

`allow`: 파일 맨 아래에 `- 금요일에는 예약 없이 쓴다.`가 더해졌습니다(2026-10-21 실행).

</details>

### 5.2 파일 고쳐 달라고 하기

In [ ]:
# ── 파일 고쳐 달라고 하기 ──
# 실행할 때마다 파일을 처음 내용으로 되돌린다
RULES.write_text("# 회의실 규칙\n- 하루 전까지 예약한다.\n", encoding="utf-8")

config = {"configurable": {"thread_id": str(uuid.uuid4())}}
request = (
    "/reference/rules.md 맨 아래에 '- 금요일에는 예약 없이 쓴다.' 한 줄을 더해 줘."
)
result = agent.invoke({"messages": [("user", request)]}, config, version="v2")

print("Agent의 답:", result.value["messages"][-1].content)
print("멈췄나요?:", bool(result.interrupts))
print("--- 실행 뒤 파일 ---")
print(RULES.read_text(encoding="utf-8"))

**볼 지점**
- `deny`이면 Agent가 쓰기 권한이 없어 고칠 수 없다고 답하고, `--- 실행 뒤 파일 ---` 아래 내용이 그대로입니다.
- 고쳐졌는지는 모델의 답이 아니라 실제 파일 내용으로 확인합니다.

## 6. 조사는 하위 Agent에게 맡기기

`subagents`에 하위 Agent를 적어 두면 메인 Agent가 일을 맡깁니다.

- 검색 도구 `search_news`는 하위 Agent `researcher`만 가집니다.
- 하위 Agent는 검색 결과를 혼자 읽고 세 줄 요약만 돌려줍니다.

**왜 배우나:** 긴 검색 결과가 메인 Agent의 대화에 쌓이면 느려지고 헷갈립니다. 하위 Agent에게 맡기면 메인에는 결론만 남습니다.

준비 코드입니다. 정해 둔 가짜 기사를 돌려주는 검색 도구 `search_news`를 만듭니다. 이 셀도 실행만 합니다.

In [ ]:
from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain_core.tools import tool

from config.llm import get_llm

load_dotenv()


@tool
def search_news(query: str) -> str:
    """반도체 업계 뉴스를 검색한다."""
    # 연습용: 정해 둔 가짜 기사를 돌려준다(약 1,500자)
    articles = [
        "메모리 업계, HBM 생산 능력 확대 계획을 잇달아 발표.",
        "AI 서버 수요로 HBM 공급 부족이 내년까지 이어질 전망.",
        "차세대 HBM은 적층 단수를 늘려 용량을 키우는 방향.",
        "패키징 공정이 HBM 수율의 핵심 변수로 떠오름.",
        "고객사들이 HBM 공급처를 여러 곳으로 나누려는 움직임.",
        "전력 효율이 HBM 경쟁의 새 기준으로 부각.",
    ]
    detail = (
        " 업계 관계자는 수요와 공급, 기술 일정, 고객사 요구를 함께 살펴야 한다고 말했다."
        * 4
    )
    return "\n".join(f"[기사 {i}] {a}{detail}" for i, a in enumerate(articles, 1))

### 6.1 하위 Agent 만들고 맡기기

In [ ]:
# ── 하위 Agent 만들고 맡기기 ──
researcher = {
    "name": "researcher",
    "description": "검색이 필요한 조사를 맡는다.",
    "system_prompt": "search_news로 검색하고 핵심만 세 줄로 요약해 돌려준다.",
    "tools": [search_news],  # 검색 도구는 하위 Agent만 가진다
}

agent = create_deep_agent(
    model=get_llm(),
    subagents=[researcher],
    system_prompt="너는 팀장 보고를 돕는 비서다. 조사는 researcher에게 맡기고 받은 내용으로 답한다.",
)

request = "HBM 시장 동향을 세 줄로 요약해 줘."
result = agent.invoke({"messages": [("user", request)]}, version="v2")

# 메인 Agent의 대화
for m in result.value["messages"]:
    print(f"[{m.type}] {m.content[:100]}")

print("\n검색 원문 길이:", len(search_news.invoke("HBM")), "자")

**볼 지점**
- 메인 Agent의 대화는 네 줄입니다: `[human]` 요청 → `[ai]` 맡기기(내용은 빈 줄) → `[tool]` 하위 Agent의 세 줄 요약 → `[ai]` 답.
- 검색 원문(약 1,300자)은 메인 대화에 없습니다. 하위 Agent 안에서만 읽혔습니다.

## 7. 정리

**이 노트북에서 한 것**
- `interrupt_on`: 도구를 쓰기 직전에 멈추고 사람이 승인·수정·거절합니다.
- `ask_user` + `respond`: Agent가 선택지로 묻고 사람이 답합니다.
- `while result.interrupts:`: 멈출 때마다 메뉴로 답을 받는 반복문입니다.
- `FilesystemPermission`: Agent가 고칠 수 없는 폴더를 정합니다.
- `subagents`: 조사는 하위 Agent에게 맡기고, 메인에는 결론만 남깁니다.

다섯 가지 모두 프롬프트가 아니라 하네스가 지키는 장치입니다.